# Phase 1 内容加载验证

这个 notebook 用来验证 `version 3.0` 在 Phase 1 中新增的三部分能力：

- 内容清单 `manifest.json` 是否存在且字段完整
- TypeScript 内容加载器是否能正确读取 `rule + story`
- 当请求 `en-US` 但当前只有中文内容时，是否能自动回退到默认语言


In [ ]:
# 这个代码块用于定位项目根目录，并准备后续会复用的工具函数。
from pathlib import Path
import json
import subprocess
import sys

repo_root = Path(r"f:\Documents\GitHub\AI_TRPG_616\version 3.0")
assert repo_root.exists(), f"未找到项目目录: {repo_root}"

def run_node_validation(locale: str) -> dict:
    # 这个辅助函数会调用 Phase 1 新增的 TypeScript 内容验证脚本，并把 JSON 结果转回 Python 对象。
    command = [
        "node",
        "--experimental-strip-types",
        ".\\apps\\server\\src\\scripts\\validateContent.ts",
        "--locale",
        locale,
    ]
    completed = subprocess.run(
        command,
        cwd=repo_root,
        check=True,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    return json.loads(completed.stdout)

print(f"Python 版本: {sys.version.split()[0]}")
print(f"项目目录: {repo_root}")


In [ ]:
# 这个代码块用于验证中文默认语言是否能正确加载 rule、story 和 catalog。
zh_result = run_node_validation("zh-CN")

print("请求语言:", zh_result["requestedLocale"])
print("实际解析语言:", zh_result["resolvedLocale"])
print("规则标题:", zh_result["selectedRule"]["title"])
print("剧本标题:", zh_result["selectedStory"]["title"])
print("规则来源文件:", zh_result["selectedRule"]["ruleSource"])
print("剧本来源文件:", zh_result["selectedStory"]["storySource"])

assert zh_result["resolvedLocale"] == "zh-CN"
assert zh_result["selectedRule"]["id"] == "VHS"
assert zh_result["selectedStory"]["id"] == "THE_SILENCE"
assert len(zh_result["catalog"]) >= 1


In [ ]:
# 这个代码块用于验证语言回退逻辑：当前没有英文正文时，请求 en-US 应回退到默认语言 zh-CN。
en_result = run_node_validation("en-US")

print("请求语言:", en_result["requestedLocale"])
print("实际解析语言:", en_result["resolvedLocale"])
print("规则预览:", en_result["selectedRule"]["rulePreview"][:80])
print("剧本预览:", en_result["selectedStory"]["storyPreview"][:80])

assert en_result["requestedLocale"] == "en-US"
assert en_result["resolvedLocale"] == "zh-CN"


In [ ]:
# 这个代码块直接读取 manifest.json，交叉验证关键字段是否完整。
rule_manifest_path = repo_root / "content" / "VHS" / "rule" / "manifest.json"
story_manifest_path = repo_root / "content" / "VHS" / "story" / "The_Silence" / "manifest.json"

rule_manifest = json.loads(rule_manifest_path.read_text(encoding="utf-8"))
story_manifest = json.loads(story_manifest_path.read_text(encoding="utf-8"))

print("Rule manifest 字段:", sorted(rule_manifest.keys()))
print("Story manifest 字段:", sorted(story_manifest.keys()))

assert rule_manifest["id"] == "VHS"
assert story_manifest["ruleId"] == "VHS"
assert story_manifest["startSceneId"] == "entry_plaza"
assert "zh-CN" in rule_manifest["availableLocales"]


In [ ]:
# 这个代码块读取正文片段，确认内容文件本身也能被 Notebook 直接检查。
rule_text = (repo_root / "content" / "VHS" / "rule" / "rule.md").read_text(encoding="utf-8")
story_text = (repo_root / "content" / "VHS" / "story" / "The_Silence" / "story.md").read_text(encoding="utf-8")

print("规则正文前 200 字符:\n")
print(rule_text[:200])
print("\n剧本正文前 200 字符:\n")
print(story_text[:200])

assert "VHS_PROMPT" in rule_text
assert "The Silence" in story_text


In [ ]:
# 这个代码块验证 3.0 的集中式语言表是否已经覆盖 2.0 中那批主要语言，并检查别名归一化。
language_command = [
    "node",
    "--experimental-strip-types",
    "--input-type=module",
    "-e",
    "import { listEnabledLanguages, normalizeLocaleCode } from './packages/shared-config/src/index.ts'; console.log(JSON.stringify({ total: listEnabledLanguages().length, aliasNormalization: { en: normalizeLocaleCode('en'), jaJP: normalizeLocaleCode('ja-JP'), zh: normalizeLocaleCode('zh') } }));",
]

language_completed = subprocess.run(
    language_command,
    cwd=repo_root,
    check=True,
    capture_output=True,
    text=True,
    encoding="utf-8",
)
language_result = json.loads(language_completed.stdout)

print("集中语言表总数:", language_result["total"])
print("别名归一化结果:", language_result["aliasNormalization"])

assert language_result["total"] >= 40
assert language_result["aliasNormalization"]["en"] == "en-US"
assert language_result["aliasNormalization"]["jaJP"] == "ja"
assert language_result["aliasNormalization"]["zh"] == "zh-CN"
